# PRD Model Inference Check

Загрузка финальной PRD-модели из MLflow и тестовый predict.

**Что делает этот ноутбук:**
1. Подключается к MLflow (локальный или remote сервер)
2. Находит run с тегом `model_stage=PRD`
3. Загружает модель (LightGBM ranker)
4. Загружает небольшой тестовый набор фичей из val_fit
5. Делает predict и выводит top-N рекомендаций для 5 пользователей
6. Проверяет совпадение feature columns

**Не требует повторного обучения** — использует только сохранённые артефакты.

In [1]:
# ── Dependencies & imports ────────────────────────────────────────────────────
from __future__ import annotations

import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# Optional: load .env
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

print("Python:", __import__("platform").python_version())
try:
    import lightgbm as lgb
    print("LightGBM:", lgb.__version__)
except ImportError:
    print("LightGBM not installed — required for inference")

try:
    import mlflow
    print("MLflow:", mlflow.__version__)
except ImportError:
    mlflow = None
    print("MLflow not installed — will try direct joblib load")

Python: 3.13.7
LightGBM: 4.6.0
MLflow: 3.12.0


In [2]:
# ── Config: paths ─────────────────────────────────────────────────────────────

def _find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists() or (p / "data" / "datamart").exists() or (p / "data").exists():
            return p
    raise FileNotFoundError("project root not found")

PROJECT_ROOT = _find_project_root()
RUN_NAME     = "final_reranker_v12_repeat__merged_with_dl_all_repeat_history"

REPORTS_DIR  = PROJECT_ROOT / "artifacts" / "reports" / RUN_NAME
MODELS_DIR   = PROJECT_ROOT / "models" / "reranker" / RUN_NAME
DATASET_DIR  = PROJECT_ROOT / "data" / "processed" / "reranker" / RUN_NAME / "dataset"

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"REPORTS_DIR  : {REPORTS_DIR}")
print(f"MODELS_DIR   : {MODELS_DIR}")
print(f"DATASET_DIR  : {DATASET_DIR}")

assert REPORTS_DIR.exists(), f"Reports dir not found: {REPORTS_DIR}"
assert MODELS_DIR.exists(),  f"Models dir not found: {MODELS_DIR}"
ALLOW_LOCAL_FALLBACK = False   # True = разрешить загрузку из MODELS_DIR если MLflow недоступен
                               # False = финальный checkpoint: только MLflow/MinIO

PROJECT_ROOT : c:\Users\denis\Documents\Repo\ozon
REPORTS_DIR  : c:\Users\denis\Documents\Repo\ozon\artifacts\reports\final_reranker_v12_repeat__merged_with_dl_all_repeat_history
MODELS_DIR   : c:\Users\denis\Documents\Repo\ozon\models\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history
DATASET_DIR  : c:\Users\denis\Documents\Repo\ozon\data\processed\reranker\final_reranker_v12_repeat__merged_with_dl_all_repeat_history\dataset


In [3]:
# ── MLflow: connect & find PRD run ────────────────────────────────────────────

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT   = os.getenv("MLFLOW_EXPERIMENT_NAME", "recsys_reranker")

# Force-load .env so real AWS_* vars from ~/.aws/credentials don't shadow MinIO creds
_env_path = PROJECT_ROOT / ".env"
if _env_path.exists():
    for _line in _env_path.read_text(encoding="utf-8").splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _, _v = _line.partition("=")
            os.environ[_k.strip()] = _v.strip().strip('"').strip("'")
    print(f"Loaded credentials from {_env_path.name}")

os.environ.setdefault("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
os.environ.setdefault("AWS_ACCESS_KEY_ID",       "minioadmin")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY",   "minioadmin")
os.environ.setdefault("MLFLOW_S3_IGNORE_TLS",    "true")

# Reset boto3 default session so MinIO creds are used (not a stale AWS session)
try:
    import boto3 as _boto3
    _boto3.setup_default_session(
        aws_access_key_id     = os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key = os.environ["AWS_SECRET_ACCESS_KEY"],
        region_name           = "us-east-1",
    )
    _s3c = _boto3.client("s3", endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"],
                         aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
                         aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
                         region_name="us-east-1")
    _buckets = [b["Name"] for b in _s3c.list_buckets().get("Buckets", [])]
    print(f"MinIO OK  : {os.environ['MLFLOW_S3_ENDPOINT_URL']}  buckets={_buckets}")
    del _s3c
except Exception as _be:
    print(f"[WARN] boto3 session init: {_be}")

prd_model_uri = None
prd_run_id    = None
mlflow_active = False

# Priority 1 — prd_model_uri.txt
_uri_file = REPORTS_DIR / "prd_model_uri.txt"
if _uri_file.exists():
    prd_model_uri = _uri_file.read_text(encoding="utf-8").strip()
    print(f"prd_model_uri (file): {prd_model_uri}")

# Extract run_id from model URI
if prd_model_uri and prd_model_uri.startswith("runs:/"):
    prd_run_id = prd_model_uri.split("/")[1]

# Priority 2 — connect MLflow, search for PRD run
if mlflow is not None:
    try:
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        mlflow.search_experiments(max_results=1)
        mlflow_active = True
        print(f"MLflow {mlflow.__version__} connected : {MLFLOW_TRACKING_URI}")
    except Exception as _e:
        print(f"MLflow unreachable: {_e}")

    if mlflow_active and prd_run_id is None:
        try:
            _runs = mlflow.search_runs(
                experiment_names=[MLFLOW_EXPERIMENT],
                filter_string="tags.model_stage = 'PRD'",
                order_by=["start_time DESC"],
                max_results=1,
            )
            if len(_runs) > 0:
                prd_run_id = _runs.iloc[0]["run_id"]
                print(f"PRD run found via search: {prd_run_id}")
                if prd_model_uri is None:
                    prd_model_uri = f"runs:/{prd_run_id}/model"
        except Exception as _se:
            print(f"MLflow search failed: {_se}")

# Priority 3 — prd_run_id.txt
if prd_run_id is None:
    _id_file = REPORTS_DIR / "prd_run_id.txt"
    if _id_file.exists():
        prd_run_id = _id_file.read_text(encoding="utf-8").strip()
        print(f"prd_run_id (file): {prd_run_id}")
    if prd_model_uri is None and prd_run_id:
        prd_model_uri = f"runs:/{prd_run_id}/model"

print()
print(f"prd_model_uri = {prd_model_uri}")
print(f"prd_run_id    = {prd_run_id}")
print(f"mlflow_active = {mlflow_active}")
assert mlflow_active, "MLflow unreachable — start server: docker compose up -d"
assert prd_run_id,    "PRD run_id not found — run 5_reranker.ipynb cell 38 first"

Loaded credentials from .env
MinIO OK  : http://localhost:9000  buckets=['mlflow-artifacts']
prd_model_uri (file): models:/m-bf7d74a062384375a8a519cea87b9c92
MLflow 3.12.0 connected : http://localhost:5000
PRD run found via search: 1fd86a4ca9034f1a8cd4314a5d647023

prd_model_uri = models:/m-bf7d74a062384375a8a519cea87b9c92
prd_run_id    = 1fd86a4ca9034f1a8cd4314a5d647023
mlflow_active = True


In [4]:
# ── Load model from MLflow / MinIO ────────────────────────────────────────────
# ALLOW_LOCAL_FALLBACK=False → notebook fails if MLflow load doesn't succeed.
# Порядок попыток:
#   (a) mlflow.lightgbm.load_model(prd_model_uri)    — MLflow Model format
#   (b) mlflow.artifacts.download_artifacts + joblib  — raw joblib artifact
#   (c) local MODELS_DIR                              — только если ALLOW_LOCAL_FALLBACK=True

import tempfile as _tmp

model        = None
model_source = None
feature_cols = None

# ── (a) MLflow model URI ──────────────────────────────────────────────────────
try:
    print(f"[a] mlflow.lightgbm.load_model({prd_model_uri})")
    model = mlflow.lightgbm.load_model(prd_model_uri)
    model_source = f"mlflow_model_uri:{prd_model_uri}"
    print(f"    ✓ loaded  {type(model).__name__}")
except Exception as _e:
    print(f"    ✗ {_e}")

# ── (b) download_artifacts → joblib ──────────────────────────────────────────
if model is None:
    _ARTIFACT_CANDIDATES = [
        "model/lgbm_ranker_t02_selection.joblib",
        "model/best_ranker.joblib",
    ]
    for _ap in _ARTIFACT_CANDIDATES:
        try:
            print(f"[b] mlflow.artifacts.download_artifacts(run_id={prd_run_id[:8]}…, path={_ap})")
            _local = mlflow.artifacts.download_artifacts(
                run_id=prd_run_id, artifact_path=_ap
            )
            model = joblib.load(_local)
            model_source = f"mlflow_artifact:{_ap}"
            print(f"    ✓ downloaded → {_local}")
            print(f"    ✓ loaded  {type(model).__name__}")
            break
        except Exception as _e:
            print(f"    ✗ {_ap}: {_e}")

# ── (c) local fallback — only if explicitly allowed ───────────────────────────
if model is None and ALLOW_LOCAL_FALLBACK:
    _candidates = [
        MODELS_DIR / "lgbm_ranker_t02_selection.joblib",
        MODELS_DIR / "best_ranker.joblib",
    ]
    for _p in _candidates:
        if _p.exists():
            print(f"[c] local fallback: {_p}")
            model = joblib.load(_p)
            model_source = f"local:{_p}"
            print(f"    loaded  {type(model).__name__}")
            break

# ── Hard fail if MLflow load failed and fallback is off ───────────────────────
if model is None:
    raise RuntimeError(
        "Model not loaded from MLflow.\n"
        f"  prd_run_id    = {prd_run_id}\n"
        f"  prd_model_uri = {prd_model_uri}\n"
        "Check that:\n"
        "  1. docker compose ps  →  recsys_mlflow healthy\n"
        "  2. MinIO bucket mlflow-artifacts has model/ artifacts\n"
        "  3. Run 5_reranker.ipynb cell 38 to log model"
    )

print(f"\nmodel_source : {model_source}")

# ── Feature columns — from MLflow artifact ────────────────────────────────────
try:
    print(f"\n[feat] download_artifacts(run_id={prd_run_id[:8]}…, path=model/feature_cols.json)")
    _feat_local = mlflow.artifacts.download_artifacts(
        run_id=prd_run_id, artifact_path="model/feature_cols.json"
    )
    _feat_raw = json.loads(Path(_feat_local).read_text(encoding="utf-8"))
    if isinstance(_feat_raw, dict):
        feature_cols = _feat_raw.get("numeric_feature_cols") or []
        print(f"    ✓ {len(feature_cols)} cols (key=numeric_feature_cols)")
    else:
        feature_cols = list(_feat_raw)
        print(f"    ✓ {len(feature_cols)} cols")
except Exception as _fe:
    print(f"    ✗ MLflow artifact not found: {_fe}")
    # fallback: read from booster
    try:
        _b = model.booster_ if hasattr(model, "booster_") else model
        feature_cols = list(_b.feature_name())
        print(f"    fallback from booster: {len(feature_cols)} cols")
    except Exception as _be:
        print(f"    booster fallback failed: {_be}")

print()
print(f"model_source : {model_source}")
print(f"model type   : {type(model).__name__}")
print(f"feature cols : {len(feature_cols) if feature_cols else 'unknown'}")

[a] mlflow.lightgbm.load_model(models:/m-bf7d74a062384375a8a519cea87b9c92)


    ✓ loaded  LGBMRanker

model_source : mlflow_model_uri:models:/m-bf7d74a062384375a8a519cea87b9c92

[feat] download_artifacts(run_id=1fd86a4c…, path=model/feature_cols.json)


    ✓ 240 cols (key=numeric_feature_cols)

model_source : mlflow_model_uri:models:/m-bf7d74a062384375a8a519cea87b9c92
model type   : LGBMRanker
feature cols : 240


In [5]:
# ── Load test features: sample from val_fit dataset ───────────────────────────
# Берём несколько шардов val_fit — это уже обработанные данные с фичами.
# test-данные не используем (они не были built в v12).

N_USERS_DEMO  = 5    # сколько пользователей показать в финальном predict
N_SHARDS_LOAD = 2    # сколько шардов загрузить

vfit_dir = DATASET_DIR / "val_fit"
assert vfit_dir.exists(), f"val_fit dir not found: {vfit_dir}\nRun 5_reranker.ipynb first."

_shard_dirs = sorted(vfit_dir.glob("user_shard=*"))[:N_SHARDS_LOAD]
assert _shard_dirs, f"No shards found in {vfit_dir}"

_dfs = []
for _sd in _shard_dirs:
    _files = sorted(_sd.glob("*.parquet"))
    if _files:
        _dfs.append(pd.read_parquet(_files[0]))

assert _dfs, "No parquet files loaded"
_sample_df = pd.concat(_dfs, ignore_index=True)

print(f"Loaded {len(_sample_df):,} rows, {_sample_df['user_id'].nunique()} users")
print(f"Columns ({len(_sample_df.columns)}): {list(_sample_df.columns[:10])} ...")

Loaded 182,547 rows, 574 users
Columns (679): ['user_id', 'item_id', 'als_rank', 'als_score', 'candidate_rank', 'cat_l0', 'cat_l0_pref', 'catalog_pref', 'catalogid', 'clip_faiss_rank'] ...


In [6]:
# ── Feature columns validation ────────────────────────────────────────────────

if feature_cols is not None:
    _missing = [f for f in feature_cols if f not in _sample_df.columns]
    _extra   = [f for f in _sample_df.columns if f not in feature_cols
                and f not in ("user_id", "item_id", "label", "candidate_rank")]

    print(f"Expected features   : {len(feature_cols)}")
    print(f"Available in data   : {sum(f in _sample_df.columns for f in feature_cols)}")
    print(f"Missing features    : {len(_missing)}")
    if _missing:
        print(f"  First 10 missing: {_missing[:10]}")

    if len(_missing) > 0:
        print(f"\nFilling {len(_missing)} missing feature cols with 0.")
        for f in _missing:
            _sample_df[f] = 0.0
    else:
        print("\nAll feature columns present ✓")

Expected features   : 240
Available in data   : 240
Missing features    : 0

All feature columns present ✓


In [7]:
# ── Predict: score candidates for demo users ──────────────────────────────────

# Select N_USERS_DEMO users (prefer users with known positives)
_users_with_label = (
    _sample_df[_sample_df.get("label", pd.Series(0, index=_sample_df.index)) == 1]["user_id"]
    .unique()[:N_USERS_DEMO]
    if "label" in _sample_df.columns
    else _sample_df["user_id"].unique()[:N_USERS_DEMO]
)
_demo_df = _sample_df[_sample_df["user_id"].isin(_users_with_label)].copy()

# Prepare feature matrix
_X = _demo_df[feature_cols].copy() if feature_cols else _demo_df.select_dtypes(include="number").copy()
for f in (_X.columns if hasattr(_X, "columns") else []):
    if _X[f].dtype == object:
        _X[f] = _X[f].astype("category")

# Score
_scores = model.predict(_X)
_demo_df["_score"] = _scores

print(f"Scored {len(_demo_df):,} rows for {_demo_df['user_id'].nunique()} users")
print(f"Score range: {_scores.min():.4f} … {_scores.max():.4f}")
print()

# Show top-10 recommendations per user
TOP_K = 10
print(f"Top-{TOP_K} recommendations per user:")
print("=" * 60)
for uid, grp in _demo_df.groupby("user_id"):
    _top = grp.nlargest(TOP_K, "_score")
    _items   = _top["item_id"].tolist() if "item_id" in _top.columns else ["?"] * TOP_K
    _scores_ = _top["_score"].tolist()
    _positives = (
        _top["label"].sum() if "label" in _top.columns else "?"
    )
    print(f"user {uid}:")
    print(f"  top-{TOP_K} items  : {_items}")
    print(f"  scores         : {[round(s,3) for s in _scores_]}")
    print(f"  positives in top-{TOP_K}: {_positives}")
    print()

Scored 1,545 rows for 5 users
Score range: 10.7443 … 17.2589

Top-10 recommendations per user:
user 12800:
  top-10 items  : [264523300, 121588785, 81136479, 73809095, 32198300, 21995036, 335007050, 284134550, 313874037, 140852436]
  scores         : [16.443, 16.353, 16.182, 16.127, 16.083, 16.064, 16.003, 15.989, 15.946, 15.943]
  positives in top-10: 0

user 1758720:
  top-10 items  : [303677636, 178553884, 124830401, 69317656, 175038904, 213354667, 32455294, 89282706, 309635119, 152211020]
  scores         : [17.044, 17.021, 16.989, 16.972, 16.937, 16.907, 16.878, 16.8, 16.793, 16.783]
  positives in top-10: 1

user 4188160:
  top-10 items  : [54200672, 49010619, 306509538, 169524392, 323808780, 229368548, 154766805, 16461218, 6240579, 155676268]
  scores         : [16.897, 16.773, 16.711, 16.689, 16.688, 16.661, 16.657, 16.647, 16.645, 16.629]
  positives in top-10: 1

user 4474880:
  top-10 items  : [273708565, 124567096, 71643645, 114339152, 302134828, 12374718, 182256764, 293856

In [8]:
# ── Final status summary ──────────────────────────────────────────────────────
print("=" * 65)
print("PRD INFERENCE CHECK — SUMMARY")
print("=" * 65)
print(f"  prd_run_id   : {prd_run_id}")
print(f"  model_source : {model_source}")
print(f"  model type   : {type(model).__name__}")
print(f"  feature cols : {len(feature_cols) if feature_cols else 'unknown'}")
print(f"  sample shape : {_demo_df.shape}")

_missing_cnt = sum(f not in _demo_df.columns for f in (feature_cols or []))
print(f"  missing feats: {_missing_cnt}")

print(f"  demo users   : {_demo_df['user_id'].nunique()}")
print(f"  demo rows    : {len(_demo_df):,}")

if "label" in _demo_df.columns:
    _ndcg_demo = 0.0
    for _uid, _grp in _demo_df.groupby("user_id"):
        _ranked = _grp.sort_values("_score", ascending=False).reset_index(drop=True)
        _hits   = _ranked["label"].values[:100]
        _dcg    = sum(h / np.log2(i + 2) for i, h in enumerate(_hits))
        _idcg   = sum(h / np.log2(i + 2) for i, h in enumerate(sorted(_hits, reverse=True))) or 1.0
        _ndcg_demo += _dcg / _idcg
    _ndcg_demo /= _demo_df["user_id"].nunique()
    print(f"  NDCG@100 demo: {_ndcg_demo:.4f}")

print()
assert model_source is not None and "local:" not in model_source, (
    f"Model must be loaded from MLflow, not local. source={model_source}"
)
print("PRD model loaded from MLflow/MinIO  ✓")
print("Inference working                   ✓")
print("=" * 65)

PRD INFERENCE CHECK — SUMMARY
  prd_run_id   : 1fd86a4ca9034f1a8cd4314a5d647023
  model_source : mlflow_model_uri:models:/m-bf7d74a062384375a8a519cea87b9c92
  model type   : LGBMRanker
  feature cols : 240
  sample shape : (1545, 680)
  missing feats: 0
  demo users   : 5
  demo rows    : 1,545
  NDCG@100 demo: 0.4252

PRD model loaded from MLflow/MinIO  ✓
Inference working                   ✓
